Checking PSD of R for DINO v1 on COCO dataset

PSD of R <=> eigenvalues of R >= 0

In [1]:
import sys
import os
import torch
import torch.nn.functional as F
from torchvision import transforms
import cv2

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [3]:
# load model
model = torch.hub.load('facebookresearch/dino:main', 'dino_vits8').to(device)

Downloading: "https://github.com/facebookresearch/dino/zipball/main" to /root/.cache/torch/hub/main.zip
Downloading: "https://dl.fbaipublicfiles.com/dino/dino_deitsmall8_pretrain/dino_deitsmall8_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dino_deitsmall8_pretrain.pth


100%|██████████| 82.7M/82.7M [00:01<00:00, 68.4MB/s]


In [4]:
model.eval()

VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 384, kernel_size=(8, 8), stride=(8, 8))
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (blocks): ModuleList(
    (0-11): 12 x Block(
      (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=384, out_features=1152, bias=True)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (drop_path): Identity()
      (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=384, out_features=1536, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=1536, out_features=384, bias=True)
        (drop): Dropout(p=0.0, inplace=False)
      )
    )
  )
  (norm): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
  (head): Identity()
)

In [5]:
!wget http://images.cocodataset.org/zips/test2017.zip
!unzip -q test2017.zip

--2026-03-28 14:19:52--  http://images.cocodataset.org/zips/test2017.zip
Resolving images.cocodataset.org (images.cocodataset.org)... 52.216.43.129, 54.231.233.201, 54.231.228.249, ...
Connecting to images.cocodataset.org (images.cocodataset.org)|52.216.43.129|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6646970404 (6.2G) [application/zip]
Saving to: ‘test2017.zip’

test2017.zip        100%[===================>]   6.19G  60.3MB/s    in 1m 49s  

2026-03-28 14:21:41 (58.3 MB/s) - ‘test2017.zip’ saved [6646970404/6646970404]



In [6]:
test_dir = "test2017"
files = os.listdir(test_dir)
len(files)

40670

In [7]:
def preprocessing(image):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image_size = (224, 224)
    blob = cv2.dnn.blobFromImage(image, 1.0/255, image_size, swapRB=True, crop=False)
    blob[0][0] = (blob[0][0] - 0.485)/0.229
    blob[0][1] = (blob[0][1] - 0.456)/0.224
    blob[0][2] = (blob[0][2] - 0.406)/0.225
    return torch.tensor(blob)

In [8]:
result = model.get_intermediate_layers(preprocessing(cv2.imread(os.path.join(test_dir, files[0]))).to(device))

In [9]:
result[0].shape

torch.Size([1, 785, 384])

In [10]:
import math
math.sqrt(784)

28.0

In [12]:
lastminW = 1e9
for i, f in enumerate(files):
    img = cv2.imread(os.path.join(test_dir, f))
    with torch.no_grad():
        blob = preprocessing(img).to(device)
        result = model.get_intermediate_layers(blob)
        feats = result[0][0][1:]
        W = torch.matmul(feats, feats.T)
        minW = W.min().item()
        d = torch.matmul(feats,feats.sum(dim=0).unsqueeze(1))
        mind = d.min().item()
    if minW < lastminW:
        lastminW = minW
        print(i,'W',minW)
    if mind <= 0:
        print(i,'d',mind,"!!!!!" if mind <= 0 else "")
    D = torch.diag(d.squeeze(-1))
    eigenvalues, eigenvectors = torch.lobpcg(A=D-W, k=2, B=D, largest=False)
    if eigenvalues[0] < -1e-5:
        print(i,eigenvalues[0],"!!!!!" if eigenvalues[0] <= 0 else "")
        break
    if i % 500 == 0:
        print(i,minW,mind,eigenvalues[0].item(),eigenvalues[1].item())

0 W -455.5456237792969
0 -455.5456237792969 660654.8125 9.26756982266852e-08 0.7608736753463745
2 W -722.2122802734375
3 W -1118.6004638671875
209 W -1148.164794921875
251 W -1428.4161376953125
500 -97.72636413574219 572847.0 2.5792735414142953e-07 0.7894124388694763
732 W -1524.1256103515625
1000 -19.74420166015625 616463.0 2.779300416477781e-07 0.7408910989761353
1500 -494.31658935546875 754560.5 2.1342785316846857e-07 0.7915745377540588
2000 -196.05252075195312 774032.5 -6.361536719623473e-08 0.8055925965309143
2500 -164.89797973632812 621111.5 -2.5556017746453108e-08 0.72971111536026
3000 -492.54937744140625 718959.25 2.7023196480513434e-07 0.6913924217224121
3500 -629.1912231445312 763831.8125 -3.1341222950231895e-08 0.7625955939292908
4000 -590.8150024414062 588743.375 5.9274700703326744e-08 0.7820353507995605
4500 -363.6289367675781 859420.25 2.558614085046429e-07 0.8079312443733215
5000 -420.08148193359375 793113.3125 7.83612250643273e-08 0.7754029035568237
5500 66.318695068359

Though the values of W are negative, d is well-defined and R is PSD for all samples